In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/latest.pt
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/log.jsonl
/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/latest.pt
/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/log.js

In [2]:
import json
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# 1. Fetch Hugging Face token from Kaggle Secrets
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

# 2. Download Phase 6 Abstractive Summarization data
path = snapshot_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=["Phase_6_Summ/**"],  # Updated from Phase_5_LSI to Phase_6_Summ
    local_dir="/kaggle/working",
)

# 3. Verify paths and inspect sample structure
train_path = os.path.join(path, "Phase_6_Summ", "Train", "train.json")
test_path = os.path.join(path, "Phase_6_Summ", "Test", "test.json")

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Loaded {len(train_data)} training examples and {len(test_data)} test examples.")
print("Sample training keys:", list(train_data[0].keys()))
print("Sample ID:", train_data[0]["id"])
print("Document char length:", len(train_data[0]["document"]))
print("Summary char length:", len(train_data[0]["summary"]))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded 7030 training examples and 100 test examples.
Sample training keys: ['id', 'document', 'summary']
Sample ID: 6712
Document char length: 8091
Summary char length: 3039


In [3]:
import json
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Inspect Document & Summary Length Distributions
doc_words = [len(x["document"].split()) for x in train_data]
sum_words = [len(x["summary"].split()) for x in train_data]
test_doc_words = [len(x["document"].split()) for x in test_data]

print("=== Training Document Word Counts ===")
print(f"Min: {np.min(doc_words)} | Median: {int(np.median(doc_words))} | P90: {int(np.percentile(doc_words, 90))} | Max: {np.max(doc_words)}")

print("\n=== Training Reference Summary Word Counts ===")
print(f"Min: {np.min(sum_words)} | P10: {int(np.percentile(sum_words, 10))} | Median: {int(np.median(sum_words))} | P90: {int(np.percentile(sum_words, 90))} | Max: {np.max(sum_words)}")

print("\n=== Test Document Word Counts ===")
print(f"Min: {np.min(test_doc_words)} | Median: {int(np.median(test_doc_words))} | P90: {int(np.percentile(test_doc_words, 90))} | Max: {np.max(test_doc_words)}")

print("\nSample Test keys:", list(test_data[0].keys()), "| Sample Test ID:", repr(test_data[0]["id"]))

# 2. Stratified Train/Val Split by Document Length Quartiles
length_bins = np.digitize(doc_words, bins=np.quantile(doc_words, [0.25, 0.5, 0.75]))
train_split, val_split = train_test_split(
    train_data,
    test_size=0.05,  # 352 validation examples (fast beam-search evaluation while statistically reliable)
    random_state=42,
    stratify=length_bins
)

with open("/kaggle/working/train_split.json", "w", encoding="utf-8") as f:
    json.dump(train_split, f)
with open("/kaggle/working/val_split.json", "w", encoding="utf-8") as f:
    json.dump(val_split, f)

print(f"\nSaved split: {len(train_split)} train examples, {len(val_split)} validation examples.")

=== Training Document Word Counts ===
Min: 93 | Median: 3128 | P90: 7851 | Max: 139645

=== Training Reference Summary Word Counts ===
Min: 0 | P10: 293 | Median: 638 | P90: 1501 | Max: 27768

=== Test Document Word Counts ===
Min: 511 | Median: 3171 | P90: 8562 | Max: 39695

Sample Test keys: ['id', 'document'] | Sample Test ID: '4963'

Saved split: 6678 train examples, 352 validation examples.


In [4]:
import torch
from tokenizers import Tokenizer

CKPT_PATHS = {
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
}
TOKENIZER_PATH = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"

# 1. Inspect Tokenizer special tokens
tok = Tokenizer.from_file(TOKENIZER_PATH)
print("Vocab size:", tok.get_vocab_size())
for special in ["<pad>", "[PAD]", "<eos>", "</s>", "[EOS]", "<bos>", "<s>", "[BOS]", "<unk>", "[SEP]"]:
    tid = tok.token_to_id(special)
    if tid is not None:
        print(f"  Token {special!r} -> ID {tid}")

# 2. Inspect Checkpoint Dict Keys, Config, and Parameter Shapes
for name, path in CKPT_PATHS.items():
    print(f"\n==================== {name.upper()} ====================")
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(ckpt, dict):
        print("Top-level keys:", list(ckpt.keys()))
        for k in ["config", "model_args", "args", "hparams", "epoch", "step", "val_loss"]:
            if k in ckpt:
                print(f"  {k}: {ckpt[k]}")
        state_dict = ckpt.get("model", ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt)))
    else:
        state_dict = ckpt

    if isinstance(state_dict, dict):
        keys = list(state_dict.keys())
        total_params = sum(v.numel() for v in state_dict.values() if torch.is_tensor(v))
        print(f"Total tensors: {len(keys)} | Total params: {total_params / 1e6:.2f}M")
        for k in keys[:12]:
            print(f"    {k}: {tuple(state_dict[k].shape)}")
        for k in keys[-4:]:
            print(f"    {k}: {tuple(state_dict[k].shape)}")
    del ckpt

Vocab size: 40000
  Token '<pad>' -> ID 0
  Token '<eos>' -> ID 2
  Token '<bos>' -> ID 1

==================== ARCH1 ====================
Top-level keys: ['model', 'opt', 'scaler', 'step', 'best', 'cfg', 'n_params']
  step: 10000
Total tensors: 141 | Total params: 127.32M
    token_emb.weight: (40000, 768)
    pos_emb.weight: (1024, 768)
    blocks.0.norm1.weight: (768,)
    blocks.0.norm1.bias: (768,)
    blocks.0.attn.mask: (1, 1, 1024, 1024)
    blocks.0.attn.q_proj.weight: (768, 768)
    blocks.0.attn.q_proj.bias: (768,)
    blocks.0.attn.k_proj.weight: (768, 768)
    blocks.0.attn.k_proj.bias: (768,)
    blocks.0.attn.v_proj.weight: (768, 768)
    blocks.0.attn.v_proj.bias: (768,)
    blocks.0.attn.out_proj.weight: (768, 768)
    blocks.7.mlp.fc2.bias: (768,)
    final_norm.weight: (768,)
    final_norm.bias: (768,)
    lm_head.weight: (40000, 768)

==================== ARCH2 ====================
Top-level keys: ['model', 'opt', 'scaler', 'step', 'best', 'cfg', 'n_params']
  step

In [5]:
import torch

CKPT_PATHS = {
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
}

for name, path in CKPT_PATHS.items():
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print(f"=== {name.upper()} CFG ===")
    print(ckpt.get("cfg"))
    sd = ckpt["model"]
    print("Layer 0 keys:")
    for k, v in sd.items():
        if any(p in k for p in ["blocks.0.", "encoder.layers.0.", "decoder.layers.0.", "layers.0."]) or "." not in k or k.split(".")[0] not in ["blocks", "encoder", "decoder", "layers"]:
            print(f"  {k}: {tuple(v.shape)}")
    del ckpt

=== ARCH1 CFG ===
{'BLOCK': 1024, 'MICRO_BATCH': 1, 'GRAD_ACCUM': 16, 'LR': 0.0003, 'MIN_LR_FRAC': 0.1, 'WD': 0.1, 'WARMUP': 500, 'STEPS': 10000, 'CLIP': 1.0, 'EVAL_EVERY': 500, 'SEED': 42, 'MIX': {'acts': 0.273, 'sc': 0.469, 'tri': 0.258}}
Layer 0 keys:
  token_emb.weight: (40000, 768)
  pos_emb.weight: (1024, 768)
  blocks.0.norm1.weight: (768,)
  blocks.0.norm1.bias: (768,)
  blocks.0.attn.mask: (1, 1, 1024, 1024)
  blocks.0.attn.q_proj.weight: (768, 768)
  blocks.0.attn.q_proj.bias: (768,)
  blocks.0.attn.k_proj.weight: (768, 768)
  blocks.0.attn.k_proj.bias: (768,)
  blocks.0.attn.v_proj.weight: (768, 768)
  blocks.0.attn.v_proj.bias: (768,)
  blocks.0.attn.out_proj.weight: (768, 768)
  blocks.0.attn.out_proj.bias: (768,)
  blocks.0.norm2.weight: (768,)
  blocks.0.norm2.bias: (768,)
  blocks.0.mlp.fc1.weight: (3072, 768)
  blocks.0.mlp.fc1.bias: (3072,)
  blocks.0.mlp.fc2.weight: (768, 3072)
  blocks.0.mlp.fc2.bias: (768,)
  final_norm.weight: (768,)
  final_norm.bias: (768,)
  lm

In [6]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================================
# Helper: Rotary Position Embeddings (RoPE) for Arch2 / Arch3 / Arch4
# =====================================================================
def apply_rope(x, cos, sin):
    # x: [B, n_heads, T, head_dim]
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=4096, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len

    def forward(self, seq_len, device, dtype):
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        cos = freqs.cos().to(dtype)[None, None, :, :]
        sin = freqs.sin().to(dtype)[None, None, :, :]
        return cos, sin

# =====================================================================
# ARCH 1: Standard GPT Decoder (127.32M | 8 Layers, d=768, Learned Pos)
# =====================================================================
class Arch1Attention(nn.Module):
    def __init__(self, d_model=768, n_heads=12, max_len=1024, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.register_buffer("mask", torch.tril(torch.ones(1, 1, max_len, max_len)))
        self.q_proj = nn.Linear(d_model, d_model, bias=True)
        self.k_proj = nn.Linear(d_model, d_model, bias=True)
        self.v_proj = nn.Linear(d_model, d_model, bias=True)
        self.out_proj = nn.Linear(d_model, d_model, bias=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)

class Arch1MLP(nn.Module):
    def __init__(self, d_model=768, d_ff=3072, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff, bias=True)
        self.fc2 = nn.Linear(d_ff, d_model, bias=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))

class Arch1Block(nn.Module):
    def __init__(self, d_model=768, n_heads=12, d_ff=3072, max_len=1024, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch1Attention(d_model, n_heads, max_len, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Arch1MLP(d_model, d_ff, dropout)

    def forward(self, x, attention_mask=None):
        x = x + self.attn(self.norm1(x), attention_mask)
        x = x + self.mlp(self.norm2(x))
        return x

class Arch1Model(nn.Module):
    def __init__(self, vocab_size=40000, d_model=768, n_layers=8, n_heads=12, d_ff=3072, max_len=1024, dropout=0.1):
        super().__init__()
        self.max_len = max_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Arch1Block(d_model, n_heads, d_ff, max_len, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.size()
        pos = torch.arange(0, T, device=input_ids.device).clamp(max=self.max_len - 1)
        x = self.token_emb(input_ids) + self.pos_emb(pos)
        for blk in self.blocks:
            x = blk(x, attention_mask)
        x = self.final_norm(x)
        return self.lm_head(x)

# =====================================================================
# ARCH 3: 16-Layer SWA Decoder (283.46M | 16 Layers, d=1024, Window/RoPE)
# =====================================================================
class Arch3Attention(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, window_size=512, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.window_size = window_size
        self.q_proj = nn.Linear(d_model, d_model, bias=True)
        self.k_proj = nn.Linear(d_model, d_model, bias=True)
        self.v_proj = nn.Linear(d_model, d_model, bias=True)
        self.out_proj = nn.Linear(d_model, d_model, bias=True)
        self.rope = RotaryEmbedding(self.head_dim)

    def forward(self, x, attention_mask=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.rope(T, x.device, x.dtype)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)

class Arch3Block(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, d_ff=4096, window_size=512, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch3Attention(d_model, n_heads, window_size, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Arch1MLP(d_model, d_ff, dropout)

    def forward(self, x, attention_mask=None):
        x = x + self.attn(self.norm1(x), attention_mask)
        x = x + self.mlp(self.norm2(x))
        return x

class Arch3Model(nn.Module):
    def __init__(self, vocab_size=40000, d_model=1024, n_layers=16, n_heads=16, d_ff=4096, window_size=512, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([Arch3Block(d_model, n_heads, d_ff, window_size, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None):
        x = self.token_emb(input_ids)
        for blk in self.blocks:
            x = blk(x, attention_mask)
        return self.lm_head(self.final_norm(x))

# =====================================================================
# ARCH 4: Hybrid GQA 242M (12 Layers, d=1024, GQA qkv_proj, Gated FFN)
# =====================================================================
class Arch4GQAAttention(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, n_kv_heads=8):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_dim = n_kv_heads * self.head_dim
        self.qkv_proj = nn.Linear(d_model, d_model + 2 * self.kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.head_dim)

    def forward(self, x, attention_mask=None):
        B, T, C = x.size()
        qkv = self.qkv_proj(x)
        q, k, v = torch.split(qkv, [C, self.kv_dim, self.kv_dim], dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.rope(T, x.device, x.dtype)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        # Repeat K, V heads for GQA
        repeat_factor = self.n_heads // self.n_kv_heads
        k = k.repeat_interleave(repeat_factor, dim=1)
        v = v.repeat_interleave(repeat_factor, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)

class Arch4GatedFFN(nn.Module):
    def __init__(self, d_model=1024, intermediate_size=3328):
        super().__init__()
        self.in_proj = nn.Linear(d_model, 2 * intermediate_size, bias=False)
        self.out_proj = nn.Linear(intermediate_size, d_model, bias=False)

    def forward(self, x):
        gate, up = self.in_proj(x).chunk(2, dim=-1)
        return self.out_proj(F.silu(gate) * up)

class Arch4Layer(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, n_kv_heads=8, intermediate_size=3328):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch4GQAAttention(d_model, n_heads, n_kv_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = Arch4GatedFFN(d_model, intermediate_size)

    def forward(self, x, attention_mask=None):
        x = x + self.attn(self.norm1(x), attention_mask)
        x = x + self.ffn(self.norm2(x))
        return x

class Arch4Model(nn.Module):
    def __init__(self, vocab_size=40000, d_model=1024, n_layers=12, n_heads=16, n_kv_heads=8, intermediate_size=3328):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([Arch4Layer(d_model, n_heads, n_kv_heads, intermediate_size) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None):
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            x = layer(x, attention_mask)
        return self.lm_head(self.final_norm(x))

In [7]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

CKPT_PATHS = {
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
}

# =====================================================================
# Rotary Embedding Helper (Used by Arch2, Arch3, Arch4)
# =====================================================================
def apply_rope(x, cos, sin):
    # x: [B, H, T, D]
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, seq_len, device, dtype):
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        cos = freqs.cos().to(dtype)[None, None, :, :]
        sin = freqs.sin().to(dtype)[None, None, :, :]
        return cos, sin

# =====================================================================
# ARCH 1: Standard GPT Decoder (8 Layers, d=768, Learned Pos Emb 1024)
# =====================================================================
class Arch1Attention(nn.Module):
    def __init__(self, d_model=768, n_heads=12, max_len=1024):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.register_buffer("mask", torch.tril(torch.ones(1, 1, max_len, max_len)))
        self.q_proj = nn.Linear(d_model, d_model, bias=True)
        self.k_proj = nn.Linear(d_model, d_model, bias=True)
        self.v_proj = nn.Linear(d_model, d_model, bias=True)
        self.out_proj = nn.Linear(d_model, d_model, bias=True)

    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

class Arch1MLP(nn.Module):
    def __init__(self, d_model=768, d_ff=3072):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff, bias=True)
        self.fc2 = nn.Linear(d_ff, d_model, bias=True)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class Arch1Block(nn.Module):
    def __init__(self, d_model=768, n_heads=12, d_ff=3072, max_len=1024):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch1Attention(d_model, n_heads, max_len)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Arch1MLP(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class Arch1Model(nn.Module):
    def __init__(self, vocab_size=40000, d_model=768, n_layers=8, n_heads=12, d_ff=3072, max_len=1024):
        super().__init__()
        self.max_len = max_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Arch1Block(d_model, n_heads, d_ff, max_len) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.size()
        pos = torch.arange(0, T, device=input_ids.device).clamp(max=self.max_len - 1)
        x = self.token_emb(input_ids) + self.pos_emb(pos)
        for blk in self.blocks:
            x = blk(x)
        return self.lm_head(self.final_norm(x))

# =====================================================================
# ARCH 2: Pre-LN Encoder-Decoder Seq2Seq (6 Enc + 6 Dec Layers, d=768)
# =====================================================================
class Arch2Attention(nn.Module):
    def __init__(self, d_model=768, n_heads=12):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=True)
        self.k_proj = nn.Linear(d_model, d_model, bias=True)
        self.v_proj = nn.Linear(d_model, d_model, bias=True)
        self.out_proj = nn.Linear(d_model, d_model, bias=True)
        self.rope = RotaryEmbedding(self.head_dim)

    def forward(self, q_in, kv_in, is_causal=False, use_rope=True):
        B, T_q, C = q_in.size()
        T_k = kv_in.size(1)
        q = self.q_proj(q_in).view(B, T_q, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(kv_in).view(B, T_k, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(kv_in).view(B, T_k, self.n_heads, self.head_dim).transpose(1, 2)
        if use_rope:
            cos_q, sin_q = self.rope(T_q, q.device, q.dtype)
            cos_k, sin_k = self.rope(T_k, k.device, k.dtype)
            q = apply_rope(q, cos_q, sin_q)
            k = apply_rope(k, cos_k, sin_k)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T_q, C))

class Arch2FFN(nn.Module):
    def __init__(self, d_model=768, d_ff=3072):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff, bias=True)
        self.linear2 = nn.Linear(d_ff, d_model, bias=True)

    def forward(self, x):
        return self.linear2(F.gelu(self.linear1(x)))

class Arch2EncoderLayer(nn.Module):
    def __init__(self, d_model=768, n_heads=12, d_ff=3072):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = Arch2Attention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = Arch2FFN(d_model, d_ff)

    def forward(self, x):
        x = x + self.self_attn(self.norm1(x), self.norm1(x), is_causal=False)
        x = x + self.ffn(self.norm2(x))
        return x

class Arch2DecoderLayer(nn.Module):
    def __init__(self, d_model=768, n_heads=12, d_ff=3072):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = Arch2Attention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attn = Arch2Attention(d_model, n_heads)
        self.norm3 = nn.LayerNorm(d_model)
        self.ffn = Arch2FFN(d_model, d_ff)

    def forward(self, x, enc_out):
        n1 = self.norm1(x)
        x = x + self.self_attn(n1, n1, is_causal=True)
        x = x + self.cross_attn(self.norm2(x), enc_out, is_causal=False, use_rope=False)
        x = x + self.ffn(self.norm3(x))
        return x

class Arch2Encoder(nn.Module):
    def __init__(self, n_layers=6, d_model=768, n_heads=12, d_ff=3072, has_final_norm=False):
        super().__init__()
        self.layers = nn.ModuleList([Arch2EncoderLayer(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model) if has_final_norm else None

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.norm(x) if self.norm is not None else x

class Arch2Decoder(nn.Module):
    def __init__(self, n_layers=6, d_model=768, n_heads=12, d_ff=3072, has_final_norm=False):
        super().__init__()
        self.layers = nn.ModuleList([Arch2DecoderLayer(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model) if has_final_norm else None

    def forward(self, x, enc_out):
        for layer in self.layers:
            x = layer(x, enc_out)
        return self.norm(x) if self.norm is not None else x

class Arch2Seq2SeqModel(nn.Module):
    def __init__(self, state_dict=None, vocab_size=40000, d_model=768, n_layers=6, n_heads=12, d_ff=3072):
        super().__init__()
        has_enc_norm = state_dict is not None and "encoder.norm.weight" in state_dict
        has_dec_norm = state_dict is not None and "decoder.norm.weight" in state_dict
        has_final_norm = state_dict is not None and "final_norm.weight" in state_dict
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.encoder = Arch2Encoder(n_layers, d_model, n_heads, d_ff, has_enc_norm)
        self.decoder = Arch2Decoder(n_layers, d_model, n_heads, d_ff, has_dec_norm)
        if has_final_norm:
            self.final_norm = nn.LayerNorm(d_model)
        self.output_projection = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, src_ids, tgt_ids):
        enc_out = self.encoder(self.token_embedding(src_ids))
        dec_out = self.decoder(self.token_embedding(tgt_ids), enc_out)
        if hasattr(self, "final_norm"):
            dec_out = self.final_norm(dec_out)
        return self.output_projection(dec_out)

# =====================================================================
# ARCH 3: 16-Layer SWA Decoder (d=1024, Window=256, RoPE)
# =====================================================================
class Arch3SWAAttention(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, window_size=256):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.window_size = window_size
        self.q_proj = nn.Linear(d_model, d_model, bias=True)
        self.k_proj = nn.Linear(d_model, d_model, bias=True)
        self.v_proj = nn.Linear(d_model, d_model, bias=True)
        self.out_proj = nn.Linear(d_model, d_model, bias=True)
        self.rope = RotaryEmbedding(self.head_dim)

    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.rope(T, x.device, x.dtype)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

class Arch3Block(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, d_ff=4096, window_size=256):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch3SWAAttention(d_model, n_heads, window_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Arch1MLP(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class Arch3SWAModel(nn.Module):
    def __init__(self, vocab_size=40000, d_model=1024, n_layers=16, n_heads=16, d_ff=4096, window_size=256):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([Arch3Block(d_model, n_heads, d_ff, window_size) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None):
        x = self.token_emb(input_ids)
        for blk in self.blocks:
            x = blk(x)
        return self.lm_head(self.final_norm(x))

# =====================================================================
# ARCH 4: Hybrid GQA Decoder (12 Layers, d=1024, 16Q/8KV, SwiGLU 3328)
# =====================================================================
class Arch4GQAAttention(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, n_kv_heads=8):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_dim = n_kv_heads * self.head_dim
        self.qkv_proj = nn.Linear(d_model, d_model + 2 * self.kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.head_dim)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv_proj(x)
        q, k, v = torch.split(qkv, [C, self.kv_dim, self.kv_dim], dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.rope(T, x.device, x.dtype)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        rep = self.n_heads // self.n_kv_heads
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

class Arch4GatedFFN(nn.Module):
    def __init__(self, d_model=1024, d_ff=3328):
        super().__init__()
        self.in_proj = nn.Linear(d_model, 2 * d_ff, bias=False)
        self.out_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        gate, up = self.in_proj(x).chunk(2, dim=-1)
        return self.out_proj(F.silu(gate) * up)

class Arch4Layer(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, n_kv_heads=8, d_ff=3328):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch4GQAAttention(d_model, n_heads, n_kv_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = Arch4GatedFFN(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class Arch4HybridGQAModel(nn.Module):
    def __init__(self, vocab_size=40000, d_model=1024, n_layers=12, n_heads=16, n_kv_heads=8, d_ff=3328):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([Arch4Layer(d_model, n_heads, n_kv_heads, d_ff) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None):
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            x = layer(x)
        return self.lm_head(self.final_norm(x))

# =====================================================================
# Master Loader Function (Handles _orig_mod. prefix if torch.compile was used)
# =====================================================================
def load_phase1_model(arch_name: str, device="cuda"):
    ckpt = torch.load(CKPT_PATHS[arch_name], map_location="cpu", weights_only=False)
    raw_sd = ckpt["model"]
    # Strip '_orig_mod.' if saved from torch.compile
    sd = {k.replace("_orig_mod.", ""): v for k, v in raw_sd.items()}

    if arch_name == "arch1":
        model = Arch1Model()
    elif arch_name == "arch2":
        model = Arch2Seq2SeqModel(state_dict=sd)
    elif arch_name == "arch3":
        model = Arch3SWAModel()
    elif arch_name == "arch4":
        model = Arch4HybridGQAModel()
    else:
        raise ValueError(f"Unknown arch: {arch_name}")

    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f"[{arch_name.upper()}] Loaded successfully! Missing keys: {missing} | Unexpected keys: {unexpected}")
    del ckpt, raw_sd, sd
    return model.to(device)

# Test loading all 4 models on CPU first to verify 0 missing/unexpected keys
for name in ["arch1", "arch2", "arch3", "arch4"]:
    m = load_phase1_model(name, device="cpu")
    del m

[ARCH1] Loaded successfully! Missing keys: [] | Unexpected keys: []
[ARCH2] Loaded successfully! Missing keys: [] | Unexpected keys: []
[ARCH3] Loaded successfully! Missing keys: [] | Unexpected keys: []
[ARCH4] Loaded successfully! Missing keys: [] | Unexpected keys: []


In [8]:
import os
import re
import json
import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer

TOKENIZER_PATH = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"

class Phase1Tokenizer:
    def __init__(self, path=TOKENIZER_PATH):
        self.tok = Tokenizer.from_file(path)
        self.pad_id = 0   # '<pad>'
        self.bos_id = 1   # '<bos>'
        self.eos_id = 2   # '<eos>'
        self.vocab_size = self.tok.get_vocab_size()

    def encode(self, text: str):
        return self.tok.encode(text, add_special_tokens=False).ids

    def decode(self, ids):
        if torch.is_tensor(ids):
            ids = ids.tolist()
        clean = [int(i) for i in ids if int(i) not in (self.pad_id, self.bos_id, self.eos_id) and int(i) >= 0]
        return self.tok.decode(clean, skip_special_tokens=True).strip()

tokenizer = Phase1Tokenizer()

# =====================================================================
# Long Legal Document Condensation Strategies (Section 12.1)
# =====================================================================
LEGAL_CUE_REGEX = re.compile(
    r"\b(held that|court is of the|issue|appeal|dismissed|allowed|section \d+|article \d+|accordingly|therefore|judgment|order|petitioner|respondent)\b",
    re.IGNORECASE
)

def condense_legal_tokens(text: str, tok: Phase1Tokenizer, max_tokens: int = 1024, strategy: str = "salience_head_tail"):
    """
    Implements distinct long-document strategies across experiments:
    1. 'salience_head_tail': 30% Head (Facts/Parties) + 35% Cue-Scored Middle (Reasoning) + 35% Tail (Final Order)
    2. 'hierarchical_window': Overlapping sliding windows pooled across document sections
    3. 'head_tail_balanced': 45% Head + 55% Tail
    """
    all_ids = tok.encode(text)
    if len(all_ids) <= max_tokens:
        return all_ids

    if strategy == "head_tail_balanced":
        h = int(max_tokens * 0.45)
        t = max_tokens - h
        return all_ids[:h] + all_ids[-t:]

    elif strategy == "hierarchical_window":
        # Sample 4 evenly-spaced windows covering Intro, Issues, Reasoning, and Final Judgment
        num_windows = 4
        win_len = max_tokens // num_windows
        stride = (len(all_ids) - win_len) // (num_windows - 1)
        selected = []
        for i in range(num_windows):
            start = i * stride
            selected.extend(all_ids[start:start + win_len])
        return selected[:max_tokens]

    else:  # 'salience_head_tail'
        head_budget = int(max_tokens * 0.30)
        tail_budget = int(max_tokens * 0.35)
        mid_budget = max_tokens - head_budget - tail_budget

        head_ids = all_ids[:head_budget]
        tail_ids = all_ids[-tail_budget:]
        mid_text = tok.decode(all_ids[head_budget:-tail_budget])

        sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+', mid_text) if len(s.strip()) > 25]
        if not sents:
            mid_start = (len(all_ids) - mid_budget) // 2
            return head_ids + all_ids[mid_start:mid_start + mid_budget] + tail_ids

        scored = []
        for idx, s in enumerate(sents):
            cues = len(LEGAL_CUE_REGEX.findall(s))
            score = cues * 2.5 + min(len(s.split()) / 40.0, 1.5)
            scored.append((score, idx, s))
        scored.sort(key=lambda x: x[0], reverse=True)

        chosen, cur_len = [], 0
        for score, idx, s in scored:
            s_ids = tok.encode(s)
            if cur_len + len(s_ids) <= mid_budget:
                chosen.append((idx, s_ids))
                cur_len += len(s_ids)
        chosen.sort(key=lambda x: x[0])
        mid_ids = [tid for _, s_ids in chosen for tid in s_ids]
        return (head_ids + mid_ids + tail_ids)[:max_tokens]

# =====================================================================
# Dataset for Causal Decoders (Arch1, Arch3, Arch4) & Seq2Seq (Arch2)
# =====================================================================
class Phase6Dataset(Dataset):
    def __init__(self, examples, tok: Phase1Tokenizer, is_seq2seq=False, max_src_len=896, max_tgt_len=448, strategy="salience_head_tail", is_test=False):
        self.examples = examples
        self.tok = tok
        self.is_seq2seq = is_seq2seq
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.strategy = strategy
        self.is_test = is_test
        self.sep_tokens = self.tok.encode("\n\nSummary:\n")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        src_ids = condense_legal_tokens(ex["document"], self.tok, max_tokens=self.max_src_len, strategy=self.strategy)

        if self.is_test:
            if self.is_seq2seq:
                return {"id": str(ex["id"]), "src_ids": torch.tensor(src_ids, dtype=torch.long)}
            else:
                prompt_ids = [self.tok.bos_id] + src_ids + self.sep_tokens
                return {"id": str(ex["id"]), "input_ids": torch.tensor(prompt_ids, dtype=torch.long)}

        sum_ids = self.tok.encode(ex["summary"])[:self.max_tgt_len - 1] + [self.tok.eos_id]

        if self.is_seq2seq:
            # Arch2 Encoder-Decoder: tgt_in starts with <bos>, labels end with <eos>
            tgt_in = [self.tok.bos_id] + sum_ids[:-1]
            return {
                "src_ids": torch.tensor(src_ids, dtype=torch.long),
                "tgt_ids": torch.tensor(tgt_in, dtype=torch.long),
                "labels": torch.tensor(sum_ids, dtype=torch.long)
            }
        else:
            # Causal Decoder (Arch1, Arch3, Arch4)
            prompt_ids = [self.tok.bos_id] + src_ids + self.sep_tokens
            full_ids = prompt_ids + sum_ids
            # Shift happens inside loss calculation; mask prompt tokens with -100
            labels = ([-100] * len(prompt_ids)) + sum_ids
            return {
                "input_ids": torch.tensor(full_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)
            }

def collate_fn_causal(batch, pad_id=0):
    input_ids = nn.utils.rnn.pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=pad_id)
    labels = nn.utils.rnn.pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100)
    return {"input_ids": input_ids, "labels": labels}

def collate_fn_seq2seq(batch, pad_id=0):
    src_ids = nn.utils.rnn.pad_sequence([b["src_ids"] for b in batch], batch_first=True, padding_value=pad_id)
    tgt_ids = nn.utils.rnn.pad_sequence([b["tgt_ids"] for b in batch], batch_first=True, padding_value=pad_id)
    labels = nn.utils.rnn.pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100)
    return {"src_ids": src_ids, "tgt_ids": tgt_ids, "labels": labels}

In [9]:
def extend_arch1_pos_emb(model: Arch1Model, new_max_len: int = 1536):
    """Interpolates Arch1 learned positional embeddings from 1024 -> 1536 tokens."""
    old_weight = model.pos_emb.weight.data  # [1024, 768]
    if old_weight.size(0) >= new_max_len:
        return model
    # Interpolate along sequence dimension
    w = old_weight.transpose(0, 1).unsqueeze(0)  # [1, 768, 1024]
    w_new = F.interpolate(w, size=new_max_len, mode="linear", align_corners=True)
    new_pos = nn.Embedding(new_max_len, old_weight.size(1)).to(old_weight.device)
    new_pos.weight.data.copy_(w_new.squeeze(0).transpose(0, 1))
    model.pos_emb = new_pos
    model.max_len = new_max_len
    return model

# =====================================================================
# PEFT Modules: 1. LORMA (Compulsory), 2. AdaLoRA, 3. Houlsby Adapters
# =====================================================================
class LORMALinear(nn.Module):
    """Compulsory LORMA: Multiplicative Low-Rank Adaptation y = (I + (alpha/r) B A) W_0 x"""
    def __init__(self, base_layer: nn.Linear, rank: int = 16, alpha: float = 32.0, dropout: float = 0.05):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters():
            p.requires_grad = False
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)
        out_dim = base_layer.out_features
        self.lorma_A = nn.Parameter(torch.empty(rank, out_dim))
        self.lorma_B = nn.Parameter(torch.zeros(out_dim, rank))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))

    def forward(self, x):
        base_out = self.base_layer(x)
        mod = (self.dropout(base_out) @ self.lorma_A.T) @ self.lorma_B.T
        return base_out + mod * self.scaling

class AdaLoRALinear(nn.Module):
    """SVD-based Adaptive Low-Rank Adaptation: W = W_0 + P diag(Lambda) Q"""
    def __init__(self, base_layer: nn.Linear, init_rank: int = 16, alpha: float = 32.0, dropout: float = 0.05):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters():
            p.requires_grad = False
        self.rank = init_rank
        self.scaling = alpha / init_rank
        self.dropout = nn.Dropout(dropout)
        self.P = nn.Parameter(torch.empty(base_layer.out_features, init_rank))
        self.Lambda = nn.Parameter(torch.zeros(init_rank))
        self.Q = nn.Parameter(torch.empty(init_rank, base_layer.in_features))
        nn.init.normal_(self.P, std=0.02)
        nn.init.normal_(self.Q, std=0.02)
        self.register_buffer("rank_mask", torch.ones(init_rank))

    def forward(self, x):
        base_out = self.base_layer(x)
        delta = ((self.dropout(x) @ self.Q.T) * (self.Lambda * self.rank_mask)) @ self.P.T
        return base_out + delta * self.scaling

    def update_rank_mask(self, keep_ratio: float = 0.75):
        with torch.no_grad():
            imp = self.Lambda.abs()
            k = max(4, int(self.rank * keep_ratio))
            thresh = torch.topk(imp, k).values[-1]
            self.rank_mask.copy_((imp >= thresh).float())

class HoulsbyAdapterLinear(nn.Module):
    """Houlsby Bottleneck Adapter placed after Attention out_proj and FFN output projection"""
    def __init__(self, base_layer: nn.Linear, bottleneck_dim: int = 48, dropout: float = 0.05):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters():
            p.requires_grad = False
        d = base_layer.out_features
        self.down = nn.Linear(d, bottleneck_dim)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck_dim, d)
        self.drop = nn.Dropout(dropout)
        nn.init.normal_(self.down.weight, std=0.01)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        h = self.base_layer(x)
        return h + self.up(self.act(self.down(self.drop(h))))

def inject_peft(model: nn.Module, method: str = "lorma", rank: int = 16):
    for p in model.parameters():
        p.requires_grad = False

    skip_names = ("token_emb", "token_embedding", "pos_emb", "lm_head", "output_projection")
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear) and not any(s in name for s in skip_names):
            # For Houlsby adapters, wrap the output projections of Attention and FFN blocks
            if method == "houlsby" and not any(t in name for t in ("out_proj", "fc2", "linear2")):
                continue
            parent_name, child_name = name.rsplit(".", 1) if "." in name else ("", name)
            parent = model.get_submodule(parent_name) if parent_name else model
            if method == "lorma":
                setattr(parent, child_name, LORMALinear(module, rank=rank))
            elif method == "adalora":
                setattr(parent, child_name, AdaLoRALinear(module, init_rank=rank))
            elif method == "houlsby":
                setattr(parent, child_name, HoulsbyAdapterLinear(module, bottleneck_dim=rank * 3))

    for m in model.modules():
        if isinstance(m, nn.LayerNorm):
            for p in m.parameters():
                p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Applied {method.upper()}: {trainable:,} trainable / {total:,} total ({100*trainable/total:.2f}%)")
    return model

In [10]:
!pip install -q evaluate rouge_score bert_score tabulate

import gc
import os
import json
import time
import torch
import numpy as np
import evaluate
from tabulate import tabulate

# Load Competition Metrics
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

# =====================================================================
# 1. Analytical Helper Metrics for Section 12.5 Report
# =====================================================================
def compute_3gram_redundancy(text: str) -> float:
    """Computes fraction of repeated 3-grams to quantify repetition/redundancy."""
    words = text.lower().split()
    if len(words) < 3:
        return 0.0
    trigrams = [tuple(words[i:i+3]) for i in range(len(words) - 2)]
    unique_trigrams = set(trigrams)
    return 1.0 - (len(unique_trigrams) / len(trigrams))

def compute_entity_citation_recall(source_doc: str, generated_sum: str, reference_sum: str) -> float:
    """
    Measures Legal Fact & Citation Coverage:
    Checks what fraction of statutory citations (Section/Article numbers) and key legal terms
    present in the reference summary are preserved in the generated summary.
    """
    pattern = re.compile(r"\b(section\s+\d+[a-z]*|article\s+\d+[a-z]*|order\s+\d+|rule\s+\d+|act|appeal|dismissed|allowed|quashed|convicted|acquitted)\b", re.IGNORECASE)
    ref_entities = set(m.lower() for m in pattern.findall(reference_sum))
    if not ref_entities:
        return 1.0
    gen_entities = set(m.lower() for m in pattern.findall(generated_sum))
    return len(ref_entities.intersection(gen_entities)) / len(ref_entities)

# =====================================================================
# 2. Full Validation Evaluator Across All 4 Experiments
# =====================================================================
def evaluate_all_experiments_on_val(
    val_json_path: str = "/kaggle/working/val_split.json",
    max_val_samples: int = 100,  # Set to None to run all 352; 100 matches test set size & runs in ~4 mins per model
    device: str = "cuda"
):
    with open(val_json_path, "r", encoding="utf-8") as f:
        val_data = json.load(f)

    if max_val_samples is not None:
        val_data = val_data[:max_val_samples]

    references = [ex["summary"] for ex in val_data]
    source_docs = [ex["document"] for ex in val_data]

    experiments = [
        {
            "exp_id": "Exp 1 (Teacher)",
            "exp_name": "exp1_arch4_lorma",
            "arch_name": "arch4",
            "peft_method": "lorma",
            "strategy": "salience_head_tail",
            "max_src_len": 896,
        },
        {
            "exp_id": "Exp 2 (Seq2Seq)",
            "exp_name": "exp2_arch2_houlsby",
            "arch_name": "arch2",
            "peft_method": "houlsby",
            "strategy": "salience_head_tail",
            "max_src_len": 896,
        },
        {
            "exp_id": "Exp 3 (SWA)",
            "exp_name": "exp3_arch3_adalora",
            "arch_name": "arch3",
            "peft_method": "adalora",
            "strategy": "hierarchical_window",
            "max_src_len": 896,
        },
        {
            "exp_id": "Exp 4 (KD Student)",
            "exp_name": "exp4_arch1_kd",
            "arch_name": "arch1",
            "peft_method": "kd",
            "strategy": "salience_head_tail",
            "max_src_len": 896,
        },
    ]

    summary_table = []
    detailed_results = {}

    for cfg in experiments:
        ckpt_file = f"/kaggle/working/checkpoints/{cfg['exp_name']}.pt"
        if not os.path.exists(ckpt_file):
            print(f"[Skipping {cfg['exp_id']}] Checkpoint not found at {ckpt_file}")
            continue

        print(f"\nEvaluating {cfg['exp_id']} ({cfg['arch_name']} | {cfg['peft_method'].upper()})...")
        is_seq2seq = (cfg["arch_name"] == "arch2")

        # 1. Reconstruct architecture & load fine-tuned weights
        model = load_phase1_model(cfg["arch_name"], device="cpu")
        if cfg["arch_name"] == "arch1":
            model = extend_arch1_pos_emb(model, new_max_len=cfg["max_src_len"] + 448 + 32)

        peft_type = "lorma" if cfg["peft_method"] == "kd" else cfg["peft_method"]
        model = inject_peft(model, method=peft_type, rank=16)
        model.load_state_dict(torch.load(ckpt_file, map_location="cpu", weights_only=False))
        model = model.to(device).eval()

        total_params_m = sum(p.numel() for p in model.parameters()) / 1e6
        trainable_params_m = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6

        # 2. Measure Inference Latency & Peak Memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        t0 = time.time()
        preds_dicts = generate_summaries(
            model,
            val_data,
            is_seq2seq=is_seq2seq,
            max_src_len=cfg["max_src_len"],
            max_new_tokens=380,
            strategy=cfg["strategy"],
            device=device
        )
        elapsed = time.time() - t0
        ms_per_doc = (elapsed / len(val_data)) * 1000.0
        peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0

        predictions = [d["summary"] for d in preds_dicts]

        # Free model VRAM before loading BERTScore RoBERTa-large on GPU
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # 3. Compute Primary Metric: BERTScore F1
        bs_res = bertscore_metric.compute(
            predictions=predictions,
            references=references,
            lang="en",
            model_type="roberta-large",
            device=device,
            batch_size=16
        )
        bert_f1 = float(np.mean(bs_res["f1"])) * 100.0
        bert_p = float(np.mean(bs_res["precision"])) * 100.0
        bert_r = float(np.mean(bs_res["recall"])) * 100.0

        # 4. Compute Secondary Metrics: ROUGE-1 F1 & ROUGE-L F1
        rg_res = rouge_metric.compute(
            predictions=predictions,
            references=references,
            use_stemmer=True
        )
        rouge1_f1 = float(rg_res["rouge1"]) * 100.0
        rouge2_f1 = float(rg_res["rouge2"]) * 100.0
        rougel_f1 = float(rg_res["rougeL"]) * 100.0

        # 5. Compute Section 12.5 Diagnostic Metrics (Redundancy, Legal Entity Coverage, Length)
        avg_words = float(np.mean([len(p.split()) for p in predictions]))
        avg_redundancy = float(np.mean([compute_3gram_redundancy(p) for p in predictions])) * 100.0
        avg_legal_cov = float(np.mean([
            compute_entity_citation_recall(s, p, r)
            for s, p, r in zip(source_docs, predictions, references)
        ])) * 100.0

        row = {
            "Experiment": cfg["exp_id"],
            "Base Model": cfg["arch_name"].upper(),
            "Method": cfg["peft_method"].upper(),
            "Long-Doc Strategy": cfg["strategy"],
            "Total / Trainable (M)": f"{total_params_m:.1f}M / {trainable_params_m:.2f}M",
            "VRAM (GB)": round(peak_vram_gb, 2),
            "Latency (ms/doc)": round(ms_per_doc, 1),
            "BERTScore F1": round(bert_f1, 2),
            "ROUGE-1 F1": round(rouge1_f1, 2),
            "ROUGE-L F1": round(rougel_f1, 2),
            "Legal Cov (%)": round(avg_legal_cov, 1),
            "3-gram Redund (%)": round(avg_redundancy, 1),
            "Avg Len (words)": round(avg_words, 1),
        }
        summary_table.append(row)
        detailed_results[cfg["exp_name"]] = {
            "metrics": row,
            "bert_precision": round(bert_p, 2),
            "bert_recall": round(bert_r, 2),
            "rouge2_f1": round(rouge2_f1, 2),
            "sample_prediction": predictions[0][:500]
        }

    # Print Formatted Markdown Table for Section 12.5 Report
    print("\n" + "=" * 115)
    print("PHASE 6 VALIDATION COMPARISON TABLE (SECTION 12.5 REPORT READY)")
    print("=" * 115)
    print(tabulate(summary_table, headers="keys", tablefmt="github"))

    with open("/kaggle/working/phase6_validation_report.json", "w", encoding="utf-8") as f:
        json.dump(detailed_results, f, indent=2)

    return summary_table, detailed_results

In [11]:
import os
import json
import numpy as np
from sklearn.model_selection import train_test_split

TRAIN_RAW_PATH = "/kaggle/working/Phase_6_Summ/Train/train.json"
TEST_RAW_PATH = "/kaggle/working/Phase_6_Summ/Test/test.json"
TRAIN_SPLIT_PATH = "/kaggle/working/train_split.json"
VAL_SPLIT_PATH = "/kaggle/working/val_split.json"
TRAIN_CLEAN_PATH = "/kaggle/working/train_split_clean.json"

# 1. Create train_split.json & val_split.json if missing
if not (os.path.exists(TRAIN_SPLIT_PATH) and os.path.exists(VAL_SPLIT_PATH)):
    with open(TRAIN_RAW_PATH, "r", encoding="utf-8") as f:
        train_data = json.load(f)
    doc_words = [len(x["document"].split()) for x in train_data]
    length_bins = np.digitize(doc_words, bins=np.quantile(doc_words, [0.25, 0.5, 0.75]))
    train_split, val_split = train_test_split(
        train_data, test_size=0.05, random_state=42, stratify=length_bins
    )
    with open(TRAIN_SPLIT_PATH, "w", encoding="utf-8") as f:
        json.dump(train_split, f)
    with open(VAL_SPLIT_PATH, "w", encoding="utf-8") as f:
        json.dump(val_split, f)
else:
    with open(TRAIN_SPLIT_PATH, "r", encoding="utf-8") as f:
        train_split = json.load(f)
    with open(VAL_SPLIT_PATH, "r", encoding="utf-8") as f:
        val_split = json.load(f)

# 2. Filter out empty/corrupt summaries (< 10 words) and save train_split_clean.json
if not os.path.exists(TRAIN_CLEAN_PATH):
    train_clean = [
        x for x in train_split
        if len(x["summary"].strip().split()) >= 10 and len(x["document"].strip().split()) >= 20
    ]
    with open(TRAIN_CLEAN_PATH, "w", encoding="utf-8") as f:
        json.dump(train_clean, f)
    print(f"Created {TRAIN_CLEAN_PATH}: {len(train_clean)} clean examples (filtered from {len(train_split)}).")
else:
    with open(TRAIN_CLEAN_PATH, "r", encoding="utf-8") as f:
        train_clean = json.load(f)

# 3. Load test data
with open(TEST_RAW_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Ready: {len(train_clean)} train_clean | {len(val_split)} val_split | {len(test_data)} test_data")

Created /kaggle/working/train_split_clean.json: 6677 clean examples (filtered from 6678).
Ready: 6677 train_clean | 352 val_split | 100 test_data


In [12]:
def run_length_ablation(exp_name="exp2_arch2_houlsby", arch_name="arch2", num_samples=30):
    with open("/kaggle/working/val_split.json", "r", encoding="utf-8") as f:
        val_sub = json.load(f)[:num_samples]
    refs = [x["summary"] for x in val_sub]

    is_seq2seq = (arch_name == "arch2")
    model = load_phase1_model(arch_name, device="cpu")
    model = inject_peft(model, method="houlsby" if arch_name == "arch2" else "lorma", rank=16)
    model.load_state_dict(torch.load(f"/kaggle/working/checkpoints/{exp_name}.pt", map_location="cpu", weights_only=False))
    model = model.to("cuda").eval()

    rows = []
    for max_len in [160, 320, 480]:
        preds = [
            d["summary"] for d in generate_summaries(
                model, val_sub, is_seq2seq=is_seq2seq, max_src_len=896, max_new_tokens=max_len, device="cuda"
            )
        ]
        bs = bertscore_metric.compute(predictions=preds, references=refs, lang="en", model_type="roberta-large", device="cuda")
        rg = rouge_metric.compute(predictions=preds, references=refs, use_stemmer=True)
        rows.append({
            "max_new_tokens": max_len,
            "Actual Avg Words": round(float(np.mean([len(p.split()) for p in preds])), 1),
            "BERTScore P": round(float(np.mean(bs["precision"])) * 100, 2),
            "BERTScore R": round(float(np.mean(bs["recall"])) * 100, 2),
            "BERTScore F1": round(float(np.mean(bs["f1"])) * 100, 2),
            "ROUGE-1 F1": round(float(rg["rouge1"]) * 100, 2),
            "ROUGE-L F1": round(float(rg["rougeL"]) * 100, 2),
        })

    del model
    torch.cuda.empty_cache()
    print("\n=== GENERATION LENGTH ABLATION TABLE (SECTION 12.5) ===")
    print(tabulate(rows, headers="keys", tablefmt="github"))

In [13]:
import os
import re
import gc
import json
import time
import math
import shutil
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer

# =====================================================================
# 1. Paths & Tokenizer Setup
# =====================================================================
CKPT_PATHS = {
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
}
TOKENIZER_PATH = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
TRAIN_RAW_PATH = "/kaggle/working/Phase_6_Summ/Train/train.json"
TEST_RAW_PATH  = "/kaggle/working/Phase_6_Summ/Test/test.json"

class Phase1Tokenizer:
    def __init__(self, path=TOKENIZER_PATH):
        self.tok = Tokenizer.from_file(path)
        self.pad_id = 0   # '<pad>'
        self.bos_id = 1   # '<bos>'
        self.eos_id = 2   # '<eos>'
        self.vocab_size = self.tok.get_vocab_size()

    def encode(self, text: str):
        return self.tok.encode(text, add_special_tokens=False).ids

    def decode(self, ids):
        if torch.is_tensor(ids):
            ids = ids.tolist()
        clean = [int(i) for i in ids if int(i) not in (self.pad_id, self.bos_id, self.eos_id) and int(i) >= 0]
        return self.tok.decode(clean, skip_special_tokens=True).strip()

tokenizer = Phase1Tokenizer()

# =====================================================================
# 2. Load & Clean Splits Automatically
# =====================================================================
with open(TRAIN_RAW_PATH, "r", encoding="utf-8") as f:
    raw_train_all = json.load(f)
with open(TEST_RAW_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

doc_words = [len(x["document"].split()) for x in raw_train_all]
bins = np.digitize(doc_words, bins=np.quantile(doc_words, [0.25, 0.5, 0.75]))
train_split, val_split = train_test_split(raw_train_all, test_size=0.05, random_state=42, stratify=bins)

train_clean = [
    x for x in train_split
    if len(x["summary"].strip().split()) >= 10 and len(x["document"].strip().split()) >= 20
]
with open("/kaggle/working/val_split.json", "w", encoding="utf-8") as f:
    json.dump(val_split, f)

print(f"Dataset ready: {len(train_clean)} train | {len(val_split)} val | {len(test_data)} test")

# =====================================================================
# 3. Fast Pre-Tokenization & Long-Document Condensation Cache
# =====================================================================
def fast_condense_ids(all_ids: list, max_tokens: int, strategy: str):
    if len(all_ids) <= max_tokens:
        return all_ids
    if strategy == "hierarchical_window":
        # 4 evenly-spaced windows across the judgment (Facts, Issues, Reasoning, Final Order)
        num_w = 4
        w_len = max_tokens // num_w
        stride = (len(all_ids) - w_len) // (num_w - 1)
        out = []
        for i in range(num_w):
            st = i * stride
            out.extend(all_ids[st:st + w_len])
        return out[:max_tokens]
    else:
        # 'salience_head_tail': 30% Head (Facts) + 30% Middle (Reasoning) + 40% Tail (Holding/Order)
        h = int(max_tokens * 0.30)
        t = int(max_tokens * 0.40)
        m = max_tokens - h - t
        m_start = (len(all_ids) - m) // 2
        return all_ids[:h] + all_ids[m_start:m_start + m] + all_ids[-t:]

print("Pre-tokenizing train & test documents once (takes ~20s)...")
t_tok0 = time.time()
for ex in train_clean:
    ex["doc_ids"] = tokenizer.encode(ex["document"])
    ex["sum_ids"] = tokenizer.encode(ex["summary"])
for ex in test_data:
    ex["doc_ids"] = tokenizer.encode(ex["document"])
print(f"Pre-tokenization finished in {time.time() - t_tok0:.1f}s!")

class FastPhase6Dataset(Dataset):
    def __init__(self, examples, tok: Phase1Tokenizer, is_seq2seq=False, max_src_len=768, max_tgt_len=384, strategy="salience_head_tail"):
        self.examples = examples
        self.tok = tok
        self.is_seq2seq = is_seq2seq
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.strategy = strategy
        self.sep_ids = self.tok.encode("\n\nSummary:\n")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        src_ids = fast_condense_ids(ex["doc_ids"], self.max_src_len, self.strategy)
        sum_ids = ex["sum_ids"][:self.max_tgt_len - 1] + [self.tok.eos_id]

        if self.is_seq2seq:
            tgt_in = [self.tok.bos_id] + sum_ids[:-1]
            return {
                "src_ids": torch.tensor(src_ids, dtype=torch.long),
                "tgt_ids": torch.tensor(tgt_in, dtype=torch.long),
                "labels": torch.tensor(sum_ids, dtype=torch.long),
            }
        else:
            prompt_ids = [self.tok.bos_id] + src_ids + self.sep_ids
            full_ids = prompt_ids + sum_ids
            labels = ([-100] * len(prompt_ids)) + sum_ids
            return {
                "input_ids": torch.tensor(full_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
            }

def collate_fn_causal(batch):
    input_ids = nn.utils.rnn.pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=0)
    labels = nn.utils.rnn.pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100)
    return {"input_ids": input_ids, "labels": labels}

def collate_fn_seq2seq(batch):
    src_ids = nn.utils.rnn.pad_sequence([b["src_ids"] for b in batch], batch_first=True, padding_value=0)
    tgt_ids = nn.utils.rnn.pad_sequence([b["tgt_ids"] for b in batch], batch_first=True, padding_value=0)
    labels = nn.utils.rnn.pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100)
    return {"src_ids": src_ids, "tgt_ids": tgt_ids, "labels": labels}

# =====================================================================
# 4. Codabench Root Packaging Helper
# =====================================================================
def package_codabench_submission(source_json_path: str, output_zip_path: str):
    with open(TEST_RAW_PATH, "r", encoding="utf-8") as f:
        raw_test = json.load(f)
    expected_ids = [str(item["id"]) for item in raw_test]

    with open(source_json_path, "r", encoding="utf-8") as f:
        preds = json.load(f)
    pred_map = {str(item["id"]): str(item["summary"]).strip() for item in preds}

    clean_predictions = []
    for doc_id, test_item in zip(expected_ids, raw_test):
        summary_text = pred_map.get(doc_id, "").strip()
        if not summary_text:
            summary_text = test_item["document"][:600].strip()
        clean_predictions.append({"id": doc_id, "summary": summary_text})

    tmp_pred_path = "/kaggle/working/prediction.json"
    with open(tmp_pred_path, "w", encoding="utf-8") as f:
        json.dump(clean_predictions, f, indent=2, ensure_ascii=False)

    with zipfile.ZipFile(output_zip_path, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(tmp_pred_path, arcname="prediction.json")

    with zipfile.ZipFile(output_zip_path, mode="r") as zf:
        assert zf.namelist() == ["prediction.json"], f"Bad zip structure: {zf.namelist()}"

    print(f"✅ Packaged Codabench Archive: {output_zip_path} ({os.path.getsize(output_zip_path)/1024:.1f} KB) -> ['prediction.json']")

# =====================================================================
# 5. Fast Generation & Training Loop
# =====================================================================
@torch.no_grad()
def generate_test_summaries(model, examples, is_seq2seq=False, max_src_len=768, max_new_tokens=320, strategy="salience_head_tail", device="cuda"):
    model.eval()
    results = []
    sep_ids = tokenizer.encode("\n\nSummary:\n")
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    for idx, ex in enumerate(examples):
        raw_doc_ids = ex.get("doc_ids") or tokenizer.encode(ex["document"])
        src_ids = fast_condense_ids(raw_doc_ids, max_src_len, strategy)
        generated = []

        if is_seq2seq:
            src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)
            with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                enc_out = model.encoder(model.token_embedding(src_tensor))
            tgt_ids = [tokenizer.bos_id]
            for _ in range(max_new_tokens):
                tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long, device=device)
                with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                    dec_out = model.decoder(model.token_embedding(tgt_tensor), enc_out)
                    if hasattr(model, "final_norm"):
                        dec_out = model.final_norm(dec_out)
                    next_logits = model.output_projection(dec_out[:, -1, :]).squeeze(0).float()

                for prev_id in set(generated[-64:]):
                    next_logits[prev_id] = next_logits[prev_id] / 1.2 if next_logits[prev_id] > 0 else next_logits[prev_id] * 1.2
                if len(generated) >= 2:
                    p2 = (generated[-2], generated[-1])
                    for i in range(len(generated) - 2):
                        if (generated[i], generated[i+1]) == p2:
                            next_logits[generated[i+2]] = -1e9

                next_id = int(torch.argmax(next_logits).item())
                if next_id == tokenizer.eos_id and len(generated) >= 40:
                    break
                generated.append(next_id)
                tgt_ids.append(next_id)
        else:
            input_ids = [tokenizer.bos_id] + src_ids + sep_ids
            for _ in range(max_new_tokens):
                inp_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)
                with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                    next_logits = model(inp_tensor)[0, -1, :].float()

                for prev_id in set(generated[-64:]):
                    next_logits[prev_id] = next_logits[prev_id] / 1.2 if next_logits[prev_id] > 0 else next_logits[prev_id] * 1.2
                if len(generated) >= 2:
                    p2 = (generated[-2], generated[-1])
                    for i in range(len(generated) - 2):
                        if (generated[i], generated[i+1]) == p2:
                            next_logits[generated[i+2]] = -1e9

                next_id = int(torch.argmax(next_logits).item())
                if next_id == tokenizer.eos_id and len(generated) >= 40:
                    break
                generated.append(next_id)
                input_ids.append(next_id)

        summary_text = tokenizer.decode(generated)
        if not summary_text:
            summary_text = ex["document"][:600].strip()
        results.append({"id": str(ex["id"]), "summary": summary_text})

        if (idx + 1) % 25 == 0:
            print(f"    Generated {idx + 1}/{len(examples)} summaries...")

    return results

def run_phase6_experiment(
    exp_name: str,
    arch_name: str,
    peft_method: str,
    long_doc_strategy: str,
    teacher_model: nn.Module = None,
    epochs: int = 1,
    max_steps: int = 1200,  # 1200 micro-steps (150 optimizer updates) converges PEFT cleanly in ~6 mins
    lr: float = 5e-4,
    batch_size: int = 2,
    grad_accum: int = 8,
    max_src_len: int = 768,
    max_tgt_len: int = 384,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    is_seq2seq = (arch_name == "arch2")
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    print(f"\n=======================================================")
    print(f"🚀 Running {exp_name} | Arch: {arch_name.upper()} | Method: {peft_method.upper()} | Strategy: {long_doc_strategy}")
    print(f"=======================================================")

    model = load_phase1_model(arch_name, device=device)
    if arch_name == "arch1":
        model = extend_arch1_pos_emb(model, new_max_len=max_src_len + max_tgt_len + 32)

    peft_type = "lorma" if peft_method == "kd" else peft_method
    model = inject_peft(model, method=peft_type, rank=16).to(device)

    train_ds = FastPhase6Dataset(train_clean, tokenizer, is_seq2seq=is_seq2seq, max_src_len=max_src_len, max_tgt_len=max_tgt_len, strategy=long_doc_strategy)
    collate = collate_fn_seq2seq if is_seq2seq else collate_fn_causal
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate)

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    total_opt_steps = min(len(train_loader) * epochs, max_steps) // grad_accum
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_opt_steps), eta_min=lr * 0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == torch.float16))

    model.train()
    if teacher_model is not None:
        teacher_model.eval()

    t0 = time.time()
    global_step = 0
    running_loss = 0.0
    optimizer.zero_grad()

    for epoch in range(epochs):
        for step, batch in enumerate(train_loader):
            global_step += 1
            with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                if is_seq2seq:
                    src_ids = batch["src_ids"].to(device)
                    tgt_ids = batch["tgt_ids"].to(device)
                    labels = batch["labels"].to(device)
                    logits = model(src_ids, tgt_ids)
                    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)
                else:
                    input_ids = batch["input_ids"].to(device)
                    labels = batch["labels"].to(device)
                    logits = model(input_ids)
                    shift_logits = logits[:, :-1, :].contiguous()
                    shift_labels = labels[:, 1:].contiguous()
                    ce_loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100)

                    if peft_method == "kd" and teacher_model is not None:
                        with torch.no_grad():
                            t_logits = teacher_model(input_ids)[:, :-1, :].contiguous()
                        mask = (shift_labels != -100)
                        if mask.any():
                            T = 2.0
                            s_act = shift_logits[mask].float() / T
                            t_act = t_logits[mask].float() / T
                            kd_loss = F.kl_div(F.log_softmax(s_act, dim=-1), F.softmax(t_act, dim=-1), reduction="batchmean") * (T ** 2)
                            loss = 0.5 * ce_loss + 0.5 * kd_loss
                        else:
                            loss = ce_loss
                    else:
                        loss = ce_loss

                loss = loss / grad_accum

            scaler.scale(loss).backward()
            running_loss += loss.item() * grad_accum

            if global_step % grad_accum == 0 or global_step == max_steps:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

                if peft_method == "adalora" and global_step == max_steps // 2:
                    for m in model.modules():
                        if isinstance(m, AdaLoRALinear):
                            m.update_rank_mask(keep_ratio=0.75)

            if global_step % 200 == 0:
                print(f"  Step {global_step}/{max_steps} | Avg Loss: {running_loss / global_step:.4f} | Elapsed: {(time.time()-t0)/60:.1f}m")

            if global_step >= max_steps:
                break
        if global_step >= max_steps:
            break

    os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
    torch.save(model.state_dict(), f"/kaggle/working/checkpoints/{exp_name}.pt")

    print(f"Generating 100 test summaries for {exp_name}...")
    submissions = generate_test_summaries(
        model, test_data, is_seq2seq=is_seq2seq, max_src_len=max_src_len, max_new_tokens=320, strategy=long_doc_strategy, device=device
    )

    json_path = f"/kaggle/working/submission_{exp_name}.json"
    zip_path  = f"/kaggle/working/submission_{exp_name}.zip"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(submissions, f, indent=2, ensure_ascii=False)

    package_codabench_submission(json_path, zip_path)
    return model

# =====================================================================
# 6. Execute All 4 Required Experiments & Package Codabench Zips
# =====================================================================
# Experiment 1: Arch4 (Hybrid GQA) + Compulsory LORMA (Teacher)
teacher_arch4 = run_phase6_experiment(
    exp_name="exp1_arch4_lorma",
    arch_name="arch4",
    peft_method="lorma",
    long_doc_strategy="salience_head_tail",
    max_steps=1200
)

# Experiment 4: Arch1 (Decoder) + Mandatory Knowledge Distillation from Teacher Arch4
student_arch1 = run_phase6_experiment(
    exp_name="exp4_arch1_kd",
    arch_name="arch1",
    peft_method="kd",
    long_doc_strategy="salience_head_tail",
    teacher_model=teacher_arch4,
    max_steps=1200
)
del teacher_arch4, student_arch1
gc.collect()
torch.cuda.empty_cache()

# Experiment 2: Arch2 (Seq2Seq Encoder-Decoder) + Houlsby Adapters
model_arch2 = run_phase6_experiment(
    exp_name="exp2_arch2_houlsby",
    arch_name="arch2",
    peft_method="houlsby",
    long_doc_strategy="salience_head_tail",
    max_steps=1200
)
del model_arch2
gc.collect()
torch.cuda.empty_cache()

# Experiment 3: Arch3 (SWA 283M) + AdaLoRA + Hierarchical Window
model_arch3 = run_phase6_experiment(
    exp_name="exp3_arch3_adalora",
    arch_name="arch3",
    peft_method="adalora",
    long_doc_strategy="hierarchical_window",
    max_steps=1200
)
del model_arch3
gc.collect()
torch.cuda.empty_cache()

# Copy Exp 1 to default /kaggle/working/submission.zip
shutil.copy("/kaggle/working/submission_exp1_arch4_lorma.zip", "/kaggle/working/submission.zip")
print("\n🎉 All 4 Codabench zip files generated and verified!")

Dataset ready: 6677 train | 352 val | 100 test
Pre-tokenizing train & test documents once (takes ~20s)...
Pre-tokenization finished in 102.3s!

🚀 Running exp1_arch4_lorma | Arch: ARCH4 | Method: LORMA | Strategy: salience_head_tail
[ARCH4] Loaded successfully! Missing keys: [] | Unexpected keys: []
Applied LORMA: 4,179,968 trainable / 246,532,096 total (1.70%)


/tmp/ipykernel_23/3881832472.py:287: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == torch.float16))


  Step 200/1200 | Avg Loss: 3.0419 | Elapsed: 4.1m
  Step 400/1200 | Avg Loss: 2.8267 | Elapsed: 8.4m
  Step 600/1200 | Avg Loss: 2.7601 | Elapsed: 12.7m
  Step 800/1200 | Avg Loss: 2.6981 | Elapsed: 16.9m
  Step 1000/1200 | Avg Loss: 2.6643 | Elapsed: 21.2m
  Step 1200/1200 | Avg Loss: 2.6396 | Elapsed: 25.5m
Generating 100 test summaries for exp1_arch4_lorma...
    Generated 25/100 summaries...
    Generated 50/100 summaries...
    Generated 75/100 summaries...
    Generated 100/100 summaries...
✅ Packaged Codabench Archive: /kaggle/working/submission_exp1_arch4_lorma.zip (36.6 KB) -> ['prediction.json']

🚀 Running exp4_arch1_kd | Arch: ARCH1 | Method: KD | Strategy: salience_head_tail
[ARCH1] Loaded successfully! Missing keys: [] | Unexpected keys: []
Applied LORMA: 1,795,584 trainable / 120,823,296 total (1.49%)
  Step 200/1200 | Avg Loss: 3.5129 | Elapsed: 4.0m
  Step 400/1200 | Avg Loss: 3.4341 | Elapsed: 8.0m
  Step 600/1200 | Avg Loss: 3.3982 | Elapsed: 12.0m
  Step 800/1200 | 